# SteelFlow: baseline model comparison

This notebook compares multiple algorithms on late-delivery classification and demand forecasting. Metrics are calculated on the supplied time-based splits.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
con = duckdb.connect()
sns.set_theme(style="whitegrid")


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingRegressor


In [ ]:
late = con.execute(f"SELECT * FROM read_parquet('{(DATA/'dispatch_delay_model_dataset.parquet').as_posix()}')").fetchdf()
late['dispatch_date'] = pd.to_datetime(late['dispatch_date'])
target = 'target_is_late'
drop = [target, 'shipment_id', 'dispatch_timestamp', 'dispatch_date', 'prediction_moment', 'synthetic_record']
X = late.drop(columns=[c for c in drop if c in late])
y = late[target].astype(int)
cat = X.select_dtypes(include=['object','bool']).columns.tolist()
num = [c for c in X.columns if c not in cat]
pre = ColumnTransformer([('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num), ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)])
models = {'logistic': LogisticRegression(max_iter=1000, class_weight='balanced'), 'random_forest': RandomForestClassifier(n_estimators=250, min_samples_leaf=5, class_weight='balanced', n_jobs=-1, random_state=42)}
results = []
for name, model in models.items():
    pipe = Pipeline([('pre', pre), ('model', model)])
    train, val, test = late['split'].eq('TRAIN'), late['split'].eq('VALIDATION'), late['split'].eq('TEST')
    pipe.fit(X[train], y[train])
    p = pipe.predict_proba(X[val])[:, 1]
    results.append({'model': name, 'pr_auc_validation': average_precision_score(y[val], p), 'roc_auc_validation': roc_auc_score(y[val], p), 'f1_validation': f1_score(y[val], p >= .5)})
pd.DataFrame(results).sort_values('pr_auc_validation', ascending=False)

## Next model extension

Add the demand regression benchmark using `forecast_features_horizon_07`, `forecast_features_horizon_14`, and `forecast_features_horizon_30`. Compare seasonal-naive lag features, Ridge, Random Forest, and gradient boosting with MAE, RMSE, and WAPE. Save only the selected pipeline and metrics after evaluation.